TO SPLIT DATASET AS REQUIRED

In [4]:
# from pathlib import Path
# import shutil
# from sklearn.model_selection import train_test_split

# # -------- PATHS --------
# data_root = Path("Data/train")
# img_dir = data_root / "image"
# ann_dir = data_root / "annos"

# out_root = Path("Data_split_exact")
# train_img_out = out_root / "train" / "image"
# train_ann_out = out_root / "train" / "annos"
# test_img_out = out_root / "test" / "image"
# test_ann_out = out_root / "test" / "annos"

# # -------- CREATE DIRS --------
# for p in [train_img_out, train_ann_out, test_img_out, test_ann_out]:
#     p.mkdir(parents=True, exist_ok=True)

# # -------- STEP 1: GET UNIQUE IMAGE NAMES --------
# # (IMPORTANT: match their logic)
# images = list(set([p.name for p in img_dir.iterdir() if p.suffix.lower() in [".jpg", ".png", ".jpeg"]]))

# # -------- STEP 2: SPLIT USING SKLEARN --------
# train_imgs, test_imgs = train_test_split(
#     images,
#     test_size=0.1,
#     random_state=42
# )

# train_set = set(train_imgs)
# test_set = set(test_imgs)

# print(f"Total: {len(images)}")
# print(f"Train: {len(train_set)}")
# print(f"Test: {len(test_set)}")

# # -------- STEP 3: COPY FILES --------
# def copy_pair(img_name, img_out_dir, ann_out_dir):
#     img_path = img_dir / img_name
#     shutil.copy(img_path, img_out_dir / img_name)

#     anno_path = ann_dir / f"{Path(img_name).stem}.json"
#     if anno_path.exists():
#         shutil.copy(anno_path, ann_out_dir / anno_path.name)

# # train
# for img in train_set:
#     copy_pair(img, train_img_out, train_ann_out)

# # test
# for img in test_set:
#     copy_pair(img, test_img_out, test_ann_out)

# print("✅ EXACT split complete!")

In [5]:
#!/usr/bin/env python3
"""
U-Net apparel segmentation for the provided DeepFashion2-style dataset.

Features:
- Discovers the top-K categories from the training split
- Builds semantic segmentation masks from polygon annotations
- Supports training from scratch or transfer learning with a ResNet34 encoder
- Uses class-weighted cross entropy + Dice loss
- Reports per-class IoU and Dice, plus macro averages
- Exports semantic masks and instance-like post-processing results
  (connected components + bounding boxes), as requested by the project brief

Expected folder layout:
Data/
  train/
    image/
    annos/
  validation/
    image/
    annos/
  test/
    image/
  json_for_validation/
  json_for_test/

The annotation parser is intentionally flexible because the exact JSON
layout can vary slightly across DeepFashion-style dumps.
"""

from __future__ import annotations

import argparse
import json
import math
import os
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.transforms import functional as TF
from tqdm import tqdm

In [6]:
# ---------------------------
# Category names from the brief
# ---------------------------
ID_TO_CATEGORY = {
    1: "short sleeve top",
    2: "long sleeve top",
    3: "short sleeve outwear",
    4: "long sleeve outwear",
    5: "vest",
    6: "sling",
    7: "shorts",
    8: "trousers",
    9: "skirt",
    10: "short sleeve dress",
    11: "long sleeve dress",
    12: "vest dress",
    13: "sling dress",
}

In [7]:
# ---------------------------
# Utility helpers
# ---------------------------
def set_seed(seed: int = 42) -> None:
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def pil_to_numpy_rgb(img: Image.Image) -> np.ndarray:
    return np.array(img.convert("RGB"))


def safe_open_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def resolve_image_and_anno_dirs(data_root: Path, split: str) -> Tuple[Path, Optional[Path]]:
    split = split.lower()
    if split == "train":
        img_dir = data_root / "train" / "image"
        ann_dir = data_root / "train" / "annos"
    elif split in {"val", "valid", "validation"}:
        img_dir = data_root / "validation" / "image"
        ann_dir = data_root / "validation" / "annos"
        if not ann_dir.exists():
            ann_dir = data_root / "json_for_validation"
    elif split == "test":
        img_dir = data_root / "test" / "image"
        ann_dir = data_root / "json_for_test"
        if not ann_dir.exists():
            ann_dir = None
    else:
        raise ValueError(f"Unknown split: {split}")
    return img_dir, ann_dir


def list_image_files(img_dir: Path) -> List[Path]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = [p for p in sorted(img_dir.iterdir()) if p.suffix.lower() in exts]
    return files


def find_json_for_image(image_path: Path, ann_dir: Optional[Path]) -> Optional[Path]:
    if ann_dir is None:
        return None
    candidate = ann_dir / f"{image_path.stem}.json"
    if candidate.exists():
        return candidate
    # fallback: search by stem
    matches = list(ann_dir.glob(f"{image_path.stem}*.json"))
    return matches[0] if matches else None


def extract_items(annotation: object) -> List[dict]:
    """
    Try to recover a list of item dictionaries from multiple possible layouts.
    Supported patterns:
    - {"items": [ ... ]}
    - {"item1": {...}, "item2": {...}}
    - [ {...}, {...} ]
    - nested dicts with category_id/category_name fields
    """
    if annotation is None:
        return []
    if isinstance(annotation, list):
        return [x for x in annotation if isinstance(x, dict)]

    if not isinstance(annotation, dict):
        return []

    # Common layouts
    if isinstance(annotation.get("items"), list):
        return [x for x in annotation["items"] if isinstance(x, dict)]

    if "annotations" in annotation and isinstance(annotation["annotations"], list):
        return [x for x in annotation["annotations"] if isinstance(x, dict)]

    # item1, item2, ...
    items = []
    for key, value in annotation.items():
        if key.lower().startswith("item") and isinstance(value, dict):
            items.append(value)

    if items:
        return items

    # fallback: any dict value that looks like an item
    for value in annotation.values():
        if isinstance(value, dict) and (
            "category_id" in value or "category_name" in value or "segmentation" in value
        ):
            items.append(value)
        elif isinstance(value, list):
            for sub in value:
                if isinstance(sub, dict) and (
                    "category_id" in sub or "category_name" in sub or "segmentation" in sub
                ):
                    items.append(sub)

    return items


def item_category_id(item: dict) -> Optional[int]:
    cid = item.get("category_id", None)
    try:
        if cid is not None:
            return int(cid)
    except Exception:
        pass

    cname = item.get("category_name")
    if isinstance(cname, str):
        for k, v in ID_TO_CATEGORY.items():
            if v == cname:
                return k
    return None


def item_polygons(item: dict) -> List[np.ndarray]:
    """
    Return a list of polygons as Nx2 arrays.
    Expected format:
      segmentation: [[x1,y1,...], [x1,y1,...], ...]
    """
    seg = item.get("segmentation", None)
    if seg is None:
        return []

    polys = []

    # Sometimes segmentation may be a flat list representing one polygon
    if isinstance(seg, list):
        if len(seg) == 0:
            return []
        if all(isinstance(x, (int, float)) for x in seg):
            coords = np.array(seg, dtype=np.float32).reshape(-1, 2)
            if len(coords) >= 3:
                polys.append(coords)
            return polys

        for poly in seg:
            if not poly:
                continue
            if isinstance(poly, list) and all(isinstance(x, (int, float)) for x in poly):
                coords = np.array(poly, dtype=np.float32).reshape(-1, 2)
                if len(coords) >= 3:
                    polys.append(coords)
            elif isinstance(poly, dict) and "points" in poly:
                pts = poly["points"]
                if isinstance(pts, list) and len(pts) >= 6:
                    coords = np.array(pts, dtype=np.float32).reshape(-1, 2)
                    if len(coords) >= 3:
                        polys.append(coords)
    return polys


def item_bbox(item: dict) -> Optional[Tuple[int, int, int, int]]:
    bb = item.get("bounding_box", item.get("bbox", None))
    if bb is None:
        return None
    if isinstance(bb, list) and len(bb) >= 4:
        x1, y1, x2, y2 = bb[:4]
        try:
            return int(x1), int(y1), int(x2), int(y2)
        except Exception:
            return None
    return None


def category_name_from_id(cid: int) -> str:
    return ID_TO_CATEGORY.get(cid, f"class_{cid}")

In [8]:
# ---------------------------
# Dataset discovery
# ---------------------------
def discover_top_k_categories(data_root: Path, train_split: str = "train", top_k: int = 5) -> List[int]:
    img_dir, ann_dir = resolve_image_and_anno_dirs(data_root, train_split)
    if ann_dir is None or not ann_dir.exists():
        raise FileNotFoundError(f"Training annotations not found in {ann_dir}")

    counts = Counter()
    for img_path in tqdm(list_image_files(img_dir), desc="Scanning train annotations"):
        anno_path = find_json_for_image(img_path, ann_dir)
        if anno_path is None or not anno_path.exists():
            continue
        try:
            ann = safe_open_json(anno_path)
        except Exception:
            continue
        for item in extract_items(ann):
            cid = item_category_id(item)
            if cid is not None and cid in ID_TO_CATEGORY:
                counts[cid] += 1

    if not counts:
        raise RuntimeError("Could not find any category annotations in the training set.")

    top = [cid for cid, _ in counts.most_common(top_k)]
    return top


def build_label_map(top_categories: Sequence[int]) -> Dict[int, int]:
    """
    Maps original category_id -> contiguous mask label index.
    0 is background, 1..K are selected apparel categories.
    """
    return {cid: i + 1 for i, cid in enumerate(top_categories)}


def build_inverse_label_map(label_map: Dict[int, int]) -> Dict[int, int]:
    return {v: k for k, v in label_map.items()}

In [9]:
# ---------------------------
# Mask drawing
# ---------------------------
def draw_item_to_mask(mask: np.ndarray, item: dict, label_idx: int) -> None:
    polys = item_polygons(item)
    if not polys:
        return
    h, w = mask.shape[:2]
    for poly in polys:
        if poly.shape[0] < 3:
            continue
        pts = np.round(poly).astype(np.int32)
        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)
        cv2.fillPoly(mask, [pts], int(label_idx))


def build_semantic_mask(image_size: Tuple[int, int], items: Sequence[dict], label_map: Dict[int, int]) -> np.ndarray:
    """
    Create a single-channel mask with values:
    0 = background
    1..K = selected classes
    """
    h, w = image_size
    mask = np.zeros((h, w), dtype=np.uint8)

    # Draw larger objects first? In semantic segmentation, later writes overwrite earlier ones.
    # Here we keep annotation order, which is the safest generic choice.
    for item in items:
        cid = item_category_id(item)
        if cid is None or cid not in label_map:
            continue
        label_idx = label_map[cid]
        draw_item_to_mask(mask, item, label_idx)
    return mask

In [10]:
# ---------------------------
# Augmentations
# ---------------------------
@dataclass
class TransformConfig:
    image_size: int = 384
    hflip_prob: float = 0.5
    normalize: bool = True


class SegmentationTransform:
    def __init__(self, cfg: TransformConfig, train: bool = True):
        self.cfg = cfg
        self.train = train

    def __call__(self, image: Image.Image, mask: np.ndarray) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.train and np.random.rand() < self.cfg.hflip_prob:
            image = TF.hflip(image)
            mask = np.ascontiguousarray(np.fliplr(mask))

        image = TF.resize(image, (self.cfg.image_size, self.cfg.image_size), interpolation=TF.InterpolationMode.BILINEAR)
        mask_pil = Image.fromarray(mask)
        mask_pil = TF.resize(mask_pil, (self.cfg.image_size, self.cfg.image_size), interpolation=TF.InterpolationMode.NEAREST)

        image_t = TF.to_tensor(image)
        if self.cfg.normalize:
            image_t = TF.normalize(image_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        mask_t = torch.from_numpy(np.array(mask_pil, dtype=np.int64))
        return image_t, mask_t

In [11]:
# ---------------------------
# Dataset
# ---------------------------
class ApparelSegmentationDataset(Dataset):
    def __init__(
        self,
        data_root: Path,
        split: str,
        label_map: Dict[int, int],
        transform: Optional[SegmentationTransform] = None,
        return_path: bool = False,
    ):
        self.data_root = Path(data_root)
        self.split = split
        self.label_map = label_map
        self.transform = transform
        self.return_path = return_path

        self.img_dir, self.ann_dir = resolve_image_and_anno_dirs(self.data_root, split)
        if not self.img_dir.exists():
            raise FileNotFoundError(f"Image directory not found: {self.img_dir}")
        self.images = list_image_files(self.img_dir)

        if split != "test" and (self.ann_dir is None or not self.ann_dir.exists()):
            raise FileNotFoundError(f"Annotation directory not found for split={split}: {self.ann_dir}")

    def __len__(self) -> int:
        return len(self.images)

    def _load_mask(self, image_path: Path, image: Image.Image) -> np.ndarray:
        if self.ann_dir is None:
            return np.zeros((image.height, image.width), dtype=np.uint8)

        anno_path = find_json_for_image(image_path, self.ann_dir)
        if anno_path is None or not anno_path.exists():
            return np.zeros((image.height, image.width), dtype=np.uint8)

        try:
            ann = safe_open_json(anno_path)
            items = extract_items(ann)
        except Exception:
            items = []

        mask = build_semantic_mask((image.height, image.width), items, self.label_map)
        return mask

    def __getitem__(self, idx: int):
        image_path = self.images[idx]
        image = Image.open(image_path).convert("RGB")
        mask = self._load_mask(image_path, image)

        if self.transform is not None:
            image_t, mask_t = self.transform(image, mask)
        else:
            image_t = TF.to_tensor(image)
            image_t = TF.normalize(image_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            mask_t = torch.from_numpy(mask.astype(np.int64))

        if self.return_path:
            return image_t, mask_t, str(image_path)
        return image_t, mask_t

In [12]:
# ---------------------------
# Model: ResNet34 U-Net
# ---------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResNet34UNet(nn.Module):
    def __init__(self, num_classes: int = 6, pretrained: bool = False):
        super().__init__()
        try:
            if pretrained:
                # Torchvision API changed across versions, so keep a safe fallback.
                try:
                    weights = models.ResNet34_Weights.DEFAULT
                    backbone = models.resnet34(weights=weights)
                except Exception:
                    backbone = models.resnet34(pretrained=True)
            else:
                backbone = models.resnet34(weights=None)
        except Exception:
            backbone = models.resnet34(weights=None)

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
        )
        self.maxpool = backbone.maxpool
        self.enc1 = backbone.layer1  # 64
        self.enc2 = backbone.layer2  # 128
        self.enc3 = backbone.layer3  # 256
        self.enc4 = backbone.layer4  # 512

        self.center = DoubleConv(512, 512)
        self.dec4 = UpBlock(512, 256, 256)
        self.dec3 = UpBlock(256, 128, 128)
        self.dec2 = UpBlock(128, 64, 64)
        self.dec1 = UpBlock(64, 64, 64)

        self.head = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x0 = self.stem(x)            # /2
        x1 = self.enc1(self.maxpool(x0))  # /4
        x2 = self.enc2(x1)           # /8
        x3 = self.enc3(x2)           # /16
        x4 = self.enc4(x3)           # /32

        c = self.center(x4)
        d4 = self.dec4(c, x3)
        d3 = self.dec3(d4, x2)
        d2 = self.dec2(d3, x1)
        d1 = self.dec1(d2, x0)
        out = self.head(d1)
        out = F.interpolate(out, scale_factor=2, mode="bilinear", align_corners=False)
        return out

In [13]:
# ---------------------------
# Losses
# ---------------------------
class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, smooth: float = 1.0, ignore_background: bool = True):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_background = ignore_background

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        probs = torch.softmax(logits, dim=1)
        target_1h = F.one_hot(target.long(), num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        start = 1 if self.ignore_background else 0
        probs = probs[:, start:]
        target_1h = target_1h[:, start:]

        dims = (0, 2, 3)
        inter = torch.sum(probs * target_1h, dims)
        denom = torch.sum(probs + target_1h, dims)
        dice = (2.0 * inter + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()

In [14]:
# ---------------------------
# Metrics
# ---------------------------
@torch.no_grad()
def fast_confusion_matrix(pred: torch.Tensor, target: torch.Tensor, num_classes: int) -> torch.Tensor:
    """
    pred, target: [N,H,W]
    returns [C,C] matrix where rows=target, cols=pred
    """
    pred = pred.view(-1).to(torch.int64)
    target = target.view(-1).to(torch.int64)
    k = (target >= 0) & (target < num_classes)
    idx = num_classes * target[k] + pred[k]
    cm = torch.bincount(idx, minlength=num_classes ** 2).reshape(num_classes, num_classes)
    return cm


@torch.no_grad()
def segmentation_metrics_from_cm(cm: torch.Tensor) -> Dict[str, object]:
    eps = 1e-7
    num_classes = cm.shape[0]
    results = {}
    per_class_iou = []
    per_class_dice = []

    for c in range(num_classes):
        tp = cm[c, c].float()
        fp = cm[:, c].sum().float() - tp
        fn = cm[c, :].sum().float() - tp
        union = tp + fp + fn
        denom = (2 * tp + fp + fn)
        iou = (tp + eps) / (union + eps)
        dice = (2 * tp + eps) / (denom + eps)
        results[f"class_{c}_iou"] = float(iou)
        results[f"class_{c}_dice"] = float(dice)
        if c != 0 and union > 0:
            per_class_iou.append(float(iou))
            per_class_dice.append(float(dice))

    results["mIoU_fg"] = float(np.mean(per_class_iou)) if per_class_iou else 0.0
    results["mDice_fg"] = float(np.mean(per_class_dice)) if per_class_dice else 0.0
    return results

In [15]:
# ---------------------------
# Class weight estimation
# ---------------------------
@torch.no_grad()
def estimate_class_weights(dataset: Dataset, num_classes: int, max_items: Optional[int] = None) -> torch.Tensor:
    counts = torch.zeros(num_classes, dtype=torch.float64)
    n = len(dataset) if max_items is None else min(len(dataset), max_items)

    for i in tqdm(range(n), desc="Estimating class weights"):
        _, mask = dataset[i]
        mask = mask.long()
        for c in range(num_classes):
            counts[c] += (mask == c).sum().item()

    counts = torch.clamp(counts, min=1.0)
    freq = counts / counts.sum()
    weights = 1.0 / torch.log1p(1.02 + freq)
    weights = weights / weights.sum() * num_classes
    weights[0] = min(weights[0].item(), 0.5)  # keep background from dominating the loss
    return weights.float()

In [16]:
# ---------------------------
# Train / eval loops
# ---------------------------
def move_batch(batch, device):
    x, y = batch
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)


def train_one_epoch(model, loader, optimizer, scaler, criterion_ce, criterion_dice, device):
    model.train()
    running = 0.0
    pbar = tqdm(loader, desc="Train", leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=scaler is not None):
            logits = model(x)
            loss_ce = criterion_ce(logits, y)
            loss_dice = criterion_dice(logits, y)
            loss = 0.7 * loss_ce + 0.3 * loss_dice

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running += loss.item() * x.size(0)
        pbar.set_postfix(loss=float(loss.item()))
    return running / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, num_classes, device):
    model.eval()
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    running_loss = 0.0
    pbar = tqdm(loader, desc="Val", leave=False)
    criterion_dice = DiceLoss(num_classes=num_classes, ignore_background=True)

    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion_dice(logits, y)
        running_loss += loss.item() * x.size(0)
        pred = torch.argmax(logits, dim=1)
        cm += fast_confusion_matrix(pred.cpu(), y.cpu(), num_classes)

    metrics = segmentation_metrics_from_cm(cm)
    metrics["val_loss"] = running_loss / len(loader.dataset)
    return metrics

In [17]:
# ---------------------------
# Prediction / post-processing
# ---------------------------
@torch.no_grad()
def predict_mask(model, image: Image.Image, image_size: int, device: torch.device) -> np.ndarray:
    model.eval()
    x = TF.resize(image.convert("RGB"), (image_size, image_size), interpolation=TF.InterpolationMode.BILINEAR)
    x = TF.to_tensor(x)
    x = TF.normalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    x = x.unsqueeze(0).to(device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

    pred = pred[0].cpu().numpy().astype(np.uint8)
    conf = conf[0].cpu().numpy()

    return (pred, conf)


def resize_mask_to_original(pred_mask: np.ndarray, original_size: Tuple[int, int]) -> np.ndarray:
    h, w = original_size
    mask_img = Image.fromarray(pred_mask)
    mask_img = mask_img.resize((w, h), resample=Image.NEAREST)
    return np.array(mask_img, dtype=np.uint8)


def postprocess_instances(
    pred_mask: np.ndarray,
    label_map: Dict[int, int],
    min_area: int = 100,
) -> List[dict]:
    inv = build_inverse_label_map(label_map)
    outputs = []

    for label_idx in sorted(inv.keys()):
        cls_mask = (pred_mask == label_idx).astype(np.uint8)
        if cls_mask.sum() == 0:
            continue

        num_cc, cc = cv2.connectedComponents(cls_mask)
        for comp_id in range(1, num_cc):
            comp = (cc == comp_id).astype(np.uint8)
            area = int(comp.sum())
            if area < min_area:
                continue

            ys, xs = np.where(comp > 0)
            x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
            category_id = inv[label_idx]
            outputs.append(
                {
                    "category_id": int(category_id),
                    "category_name": category_name_from_id(category_id),
                    "bbox": [x1, y1, x2, y2],
                    "area": area,
                }
            )
    return outputs


def save_color_mask(mask: np.ndarray, out_path: Path) -> None:
    # Save class indices as a PNG (easy to reload later)
    Image.fromarray(mask.astype(np.uint8)).save(out_path)


def colorize_mask(mask: np.ndarray, label_map: Dict[int, int]) -> np.ndarray:
    """
    Helpful for qualitative visualization.
    """
    palette = np.array(
        [
            [0, 0, 0],
            [220, 20, 60],
            [0, 128, 255],
            [0, 200, 0],
            [255, 165, 0],
            [148, 0, 211],
            [255, 215, 0],
        ],
        dtype=np.uint8,
    )
    colored = palette[np.clip(mask, 0, len(palette) - 1)]
    return colored

In [18]:
# ---------------------------
# Main commands
# ---------------------------
def cmd_train(args):
    data_root = Path(args.data_root)
    device = torch.device(args.device if torch.cuda.is_available() or args.device == "cpu" else "cpu")
    set_seed(args.seed)


    outdir = Path(args.outdir)
    ensure_dir(outdir)

    meta_cache_path = outdir / "dataset_meta.json"

    if meta_cache_path.exists():
        print("⚡ Loading cached dataset metadata...")
        with meta_cache_path.open("r") as f:
            meta = json.load(f)
        
        top_categories = meta["top_categories"]
        class_weights = torch.tensor(meta["class_weights"])
        cfg = TransformConfig(image_size=args.image_size, hflip_prob=args.hflip_prob)
        label_map = build_label_map(top_categories)
        num_classes = len(label_map) + 1
        train_ds = ApparelSegmentationDataset(
            data_root=data_root,
            split="train",
            label_map=label_map,
            transform=SegmentationTransform(cfg, train=True),
        )
        val_ds = ApparelSegmentationDataset(
            data_root=data_root,
            split="validation",
            label_map=label_map,
            transform=SegmentationTransform(cfg, train=False),
        )
        print("\nClass weights:", class_weights.tolist())

    else:
        print("🔍 Computing dataset metadata...")

        top_categories = discover_top_k_categories(data_root, "train", top_k=args.top_k)
        label_map = build_label_map(top_categories)
        num_classes = len(label_map) + 1

        print("\nSelected top categories:")
        for cid in top_categories:
            print(f"  {cid}: {category_name_from_id(cid)}")
        print("Label map (mask index -> category):")
        for cid, idx in label_map.items():
            print(f"  {idx} <- {cid}: {category_name_from_id(cid)}")

        cfg = TransformConfig(image_size=args.image_size, hflip_prob=args.hflip_prob)
        train_ds = ApparelSegmentationDataset(
            data_root=data_root,
            split="train",
            label_map=label_map,
            transform=SegmentationTransform(cfg, train=True),
        )
        val_ds = ApparelSegmentationDataset(
            data_root=data_root,
            split="validation",
            label_map=label_map,
            transform=SegmentationTransform(cfg, train=False),
        )

        class_weights = estimate_class_weights(train_ds, num_classes=num_classes, max_items=args.weight_scan_limit)
        print("\nClass weights:", class_weights.tolist())

        meta_cache_path = outdir / "dataset_meta.json"

        # Save for future runs
        with meta_cache_path.open("w") as f:
            json.dump(
                {
                    "top_categories": top_categories,
                    "class_weights": class_weights.tolist(),
                },
                f,
                indent=2,
            )

        print(f"Saved dataset metadata to {meta_cache_path}")

    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=True,
    )

    model = ResNet34UNet(num_classes=num_classes, pretrained=args.pretrained).to(device)

    criterion_ce = nn.CrossEntropyLoss(weight=class_weights.to(device))
    criterion_dice = DiceLoss(num_classes=num_classes, ignore_background=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" and args.amp else None

    best_path = outdir / "best.pt"
    meta_path = outdir / "meta.json"

    best_score = -1.0
    history = []

    
    start_epoch = 1
    best_score = -1.0

    last_ckpt_path = outdir / "last.pt"

    if last_ckpt_path.exists():
        print(f"🔁 Auto-resuming from {last_ckpt_path}")
        ckpt = torch.load(last_ckpt_path, map_location=device)

        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])

        start_epoch = ckpt["epoch"] + 1
        best_score = ckpt.get("best_score", -1.0)

        print(f"Resumed at epoch {start_epoch} with best_score={best_score:.4f}")
    else:
        print("🚀 Starting fresh training")



    for epoch in range(start_epoch, args.epochs + 1):
        print(f"\nEpoch {epoch}/{args.epochs}")
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion_ce, criterion_dice, device)
        val_metrics = evaluate(model, val_loader, num_classes, device)
        scheduler.step()

        score = val_metrics["mIoU_fg"]
        print(f"train_loss={train_loss:.4f} | val_loss={val_metrics['val_loss']:.4f} | mIoU_fg={score:.4f} | mDice_fg={val_metrics['mDice_fg']:.4f}")

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                **val_metrics,
                "lr": scheduler.get_last_lr()[0],
            }
        )

        if score > best_score:
            best_score = score
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "label_map": label_map,
                    "top_categories": top_categories,
                    "image_size": args.image_size,
                    "class_weights": class_weights.tolist(),
                    "args": vars(args),
                },
                best_path,
            )
            print(f"Saved best checkpoint to: {best_path}")

            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "best_score": best_score,
                    "label_map": label_map,
                    "top_categories": top_categories,
                    "image_size": args.image_size,
                    "class_weights": class_weights.tolist(),
                    "args": vars(args),
                },
                outdir / "last.pt",
            )

    with meta_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "label_map": label_map,
                "top_categories": top_categories,
                "num_classes": num_classes,
                "best_mIoU_fg": best_score,
                "history": history,
            },
            f,
            indent=2,
        )

    print(f"\nTraining complete. Best checkpoint: {best_path}")
    print(f"Metadata saved to: {meta_path}")

MORE HELPING COMMANDS

In [19]:
def load_checkpoint(checkpoint_path: Path, device: torch.device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    label_map = {int(k): int(v) for k, v in ckpt["label_map"].items()}
    image_size = int(ckpt.get("image_size", 384))
    num_classes = len(label_map) + 1
    model = ResNet34UNet(num_classes=num_classes, pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, label_map, image_size, ckpt


def cmd_evaluate(args):
    data_root = Path(args.data_root)
    device = torch.device(args.device if torch.cuda.is_available() or args.device == "cpu" else "cpu")
    model, label_map, image_size, ckpt = load_checkpoint(Path(args.checkpoint), device)

    cfg = TransformConfig(image_size=image_size, hflip_prob=0.0)
    val_ds = ApparelSegmentationDataset(
        data_root=data_root,
        split=args.split,
        label_map=label_map,
        transform=SegmentationTransform(cfg, train=False),
    )
    loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=args.workers)

    metrics = evaluate(model, loader, num_classes=len(label_map) + 1, device=device)

    print("\nEvaluation metrics")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    out_path = Path(args.out_metrics)
    ensure_dir(out_path.parent)
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"Saved metrics to {out_path}")


def cmd_predict(args):
    device = torch.device(args.device if torch.cuda.is_available() or args.device == "cpu" else "cpu")
    model, label_map, image_size, ckpt = load_checkpoint(Path(args.checkpoint), device)
    inv = build_inverse_label_map(label_map)

    input_dir = Path(args.input_dir)
    outdir = Path(args.outdir)
    ensure_dir(outdir)
    ensure_dir(outdir / "masks")
    ensure_dir(outdir / "instances")
    ensure_dir(outdir / "viz")

    image_files = list_image_files(input_dir)
    if not image_files:
        raise FileNotFoundError(f"No images found in {input_dir}")

    for img_path in tqdm(image_files, desc="Predict"):
        image = Image.open(img_path).convert("RGB")
        (pred, conf) = predict_mask(model, image, image_size=image_size, device=device)
        pred = resize_mask_to_original(pred, (image.height, image.width))

        # Save semantic mask
        save_color_mask(pred, outdir / "masks" / f"{img_path.stem}_mask.png")

        # Save instance-like JSON after connected components
        instances = postprocess_instances(pred, label_map=label_map, min_area=args.min_area)
        with (outdir / "instances" / f"{img_path.stem}.json").open("w", encoding="utf-8") as f:
            json.dump(
                {
                    "image": img_path.name,
                    "instances": instances,
                },
                f,
                indent=2,
            )

            # --- Color mask ---
            vis = colorize_mask(pred, label_map)

            # --- Overlay ---
            orig = np.array(image)
            alpha = 0.5
            overlay = (orig * (1 - alpha) + vis * alpha).astype(np.uint8)

            # --- Draw bounding boxes ---
            instances = postprocess_instances(pred, label_map=label_map, min_area=args.min_area)

            overlay_boxes = overlay.copy()

            for inst in instances:
                x1, y1, x2, y2 = inst["bbox"]
                label = inst["category_name"]

                # get confidence for region
                region_conf = conf[y1:y2+1, x1:x2+1]
                score = float(region_conf.mean()) if region_conf.size > 0 else 0.0

                text = f"{label} ({score:.2f})"

                cv2.rectangle(overlay_boxes, (x1, y1), (x2, y2), (255, 255, 255), 2)

                cv2.putText(
                    overlay_boxes,
                    text,
                    (x1, max(y1 - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 255, 255),
                    1,
                    cv2.LINE_AA,
                )

            # --- Combine everything ---
            combined = np.concatenate([orig, overlay_boxes], axis=1)

            # add legend below
                        # --- Create legend ---
            legend_items = []
            palette = [
                [220, 20, 60],
                [0, 128, 255],
                [0, 200, 0],
                [255, 165, 0],
                [148, 0, 211],
            ]

            for cid, idx in label_map.items():
                color = palette[idx - 1]
                name = category_name_from_id(cid)
                legend_items.append((name, color))

            # --- Create legend canvas ---
            legend_height = 40
            legend = np.ones((legend_height, combined.shape[1], 3), dtype=np.uint8) * 255

            x_offset = 20

            for name, color in legend_items:
                # color box
                cv2.rectangle(legend, (x_offset, 10), (x_offset + 20, 30), color, -1)

                # text
                cv2.putText(
                    legend,
                    name,
                    (x_offset + 30, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 0, 0),
                    1,
                    cv2.LINE_AA,
                )

                x_offset += 150
                
            # Resize legend to match combined width
            legend_resized = cv2.resize(legend, (combined.shape[1], legend.shape[0]))

            combined_with_legend = np.concatenate([combined, legend_resized], axis=0)

            # --- Save ---
            Image.fromarray(vis).save(outdir / "viz" / f"{img_path.stem}_mask.png")
            Image.fromarray(overlay).save(outdir / "viz" / f"{img_path.stem}_overlay.png")
            Image.fromarray(overlay_boxes).save(outdir / "viz" / f"{img_path.stem}_bbox.png")
            Image.fromarray(combined_with_legend).save(outdir / "viz" / f"{img_path.stem}_final.png")

            print(f"Saved predictions to: {outdir}")

ACCEPT INPUTS

In [20]:
def build_parser():
    p = argparse.ArgumentParser(description="U-Net apparel segmentation")
    sub = p.add_subparsers(dest="cmd", required=True)

    train_p = sub.add_parser("train", help="Train the model")
    train_p.add_argument("--data-root", type=str, default="Data")
    train_p.add_argument("--outdir", type=str, default="runs/unet")
    train_p.add_argument("--top-k", type=int, default=5)
    train_p.add_argument("--image-size", type=int, default=384)
    train_p.add_argument("--batch-size", type=int, default=4)
    train_p.add_argument("--epochs", type=int, default=25)
    train_p.add_argument("--lr", type=float, default=3e-4)
    train_p.add_argument("--weight-decay", type=float, default=1e-4)
    train_p.add_argument("--workers", type=int, default=4)
    train_p.add_argument("--seed", type=int, default=42)
    train_p.add_argument("--device", type=str, default="cuda")
    train_p.add_argument("--pretrained", action="store_true", help="Use ImageNet-pretrained ResNet34 encoder")
    train_p.add_argument("--amp", action="store_true", help="Use mixed precision training")
    train_p.add_argument("--hflip-prob", type=float, default=0.5)
    train_p.add_argument("--weight-scan-limit", type=int, default=None, help="Limit images scanned for class weights")
    train_p.set_defaults(func=cmd_train)

    eval_p = sub.add_parser("eval", help="Evaluate a checkpoint")
    eval_p.add_argument("--data-root", type=str, default="Data")
    eval_p.add_argument("--checkpoint", type=str, required=True)
    eval_p.add_argument("--split", type=str, default="validation", choices=["train", "validation", "val"])
    eval_p.add_argument("--batch-size", type=int, default=4)
    eval_p.add_argument("--workers", type=int, default=8)
    eval_p.add_argument("--device", type=str, default="cuda")
    eval_p.add_argument("--out-metrics", type=str, default="runs/unet/val_metrics.json")
    eval_p.set_defaults(func=cmd_evaluate)

    pred_p = sub.add_parser("predict", help="Predict masks on unlabeled images")
    pred_p.add_argument("--input-dir", type=str, required=True)
    pred_p.add_argument("--checkpoint", type=str, required=True)
    pred_p.add_argument("--outdir", type=str, default="predictions")
    pred_p.add_argument("--device", type=str, default="cuda")
    pred_p.add_argument("--min-area", type=int, default=100)
    pred_p.set_defaults(func=cmd_predict)

    return p


def main():
    parser = build_parser()
    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] {train,eval,predict} ...
ipykernel_launcher.py: error: the following arguments are required: cmd


SystemExit: 2

/home/yashubg/.local/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
%%bash
#TO SATISFY REQUIREMENTS:
pip install -r requirements_unet.txt

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
%%bash
#TO SPLIT THE DATASET:
python3 data_splitter.py

In [ ]:
%%bash
#TO TRAIN ON SPLIT DATASET ---> TRANSFER LEARNING:
python3 unet_apparel.py train \
 --data-root Data_split_exact \
 --outdir runs/unet_transferLearn \
 --epochs 12 \
 --batch-size 16 \
 --image-size 384 \
 --amp \
 --pretrained \
 --workers 8 \
 --weight-scan-limit 10000 # This line can be ignored if limit not needed

In [ ]:
%%bash
#TO TRAIN ON SPLIT DATASET ---> FROM SCRATCH: MORE EPOCHS SHOULD BE USED:
python3 unet_apparel.py train \
 --data-root Data_split_exact \
 --outdir runs/unet_scratch \
 --epochs 20 \
 --batch-size 16 \
 --image-size 384 \
 --amp \
 --workers 8 \
 --weight-scan-limit 10000 # This line can be ignored if limit not needed

In [ ]:
%%bash
#TO EVALUATE ON VALIDATION SET:
python3 unet_apparel.py eval \
 --data-root Data_split_exact \
 --checkpoint runs/unet_scratch/best.pt \
 --split validation


Evaluation metrics
class_0_iou: 0.9407
class_0_dice: 0.9695
class_1_iou: 0.7036
class_1_dice: 0.8260
class_2_iou: 0.7467
class_2_dice: 0.8550
class_3_iou: 0.6474
class_3_dice: 0.7860
class_4_iou: 0.5432
class_4_dice: 0.7040
class_5_iou: 0.6521
class_5_dice: 0.7894
mIoU_fg: 0.6586
mDice_fg: 0.7921
val_loss: 0.7084
Saved metrics to runs/unet/val_metrics.json


In [ ]:
%%bash
#TO EVALUATE ON SPLIT DATASET'S TEST SET:
python3 unet_apparel.py eval \
 --data-root Data_split_exact \
 --checkpoint runs/unet_scratch/best.pt \
 --split test


Evaluation metrics
class_0_iou: 0.8045
class_0_dice: 0.8916
class_1_iou: 0.0000
class_1_dice: 0.0000
class_2_iou: 0.0000
class_2_dice: 0.0000
class_3_iou: 0.0000
class_3_dice: 0.0000
class_4_iou: 0.0000
class_4_dice: 0.0000
class_5_iou: 0.0000
class_5_dice: 0.0000
mIoU_fg: 0.0000
mDice_fg: 0.0000
val_loss: 0.9896
Saved metrics to runs/unet/val_metrics.json


In [ ]:
%%bash
#TO SAVE METRICS (of test dataset's results) TO A FILE
python3 unet_apparel.py eval \
 --data-root Data_split_exact \
 --checkpoint runs/unet_scratch/best.pt \
 --split test \
 --out-metrics runs/unet_scratch/test_metrics.json

In [ ]:
%%bash
#COMPARE BEST MODEL VS LAST MODEL:

# Best model

python3 unet_apparel.py eval \
 --data-root Data_split_exact \
 --checkpoint runs/unet_scratch/best.pt \
 --split validation

# Last model

python3 unet_apparel.py eval \
 --data-root Data_split_exact \
 --checkpoint runs/unet_scratch/last.pt \
 --split validation

The following cell can be used for prediction task for all images in the test set

In [ ]:
%%bash
#FINALLY, PREDICT IMAGES IN THE TEST SET:
python3 unet_apparel.py predict \
 --input-dir Data_split_exact/test/image \
 --checkpoint runs/unet_scratch/best.pt \
 --outdir predictions_final

In [ ]:
%%bash
#PREDICT ON VALIDATION SET (FOR INSPECTION)
python3 unet_apparel.py predict \
 --input-dir Data_split_exact/validation/image \
 --checkpoint runs/unet_scratch/best.pt \
 --outdir predictions_val

In [ ]:
#OUTPUT STRUCTURE:

# predictions*final/
# ├── masks/
# ├── instances/
# ├── viz/
# │ ├── *\_mask.png
# │ ├── _\_overlay.png
# │ ├── _\_bbox.png
# │ ├── \_\_final.png ← ⭐ USE THIS

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix
import seaborn as sns

# -----------------------------
# 1. PRECISION-RECALL CURVE
# -----------------------------
def plot_pr_curve(y_true, y_scores, class_name="class"):
    precision, recall, _ = precision_recall_curve(y_true, y_scores)

    plt.figure()
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"PR Curve - {class_name}")
    plt.grid()
    plt.show()


# -----------------------------
# 2. CONFIDENCE VS PRECISION/RECALL
# -----------------------------
def plot_confidence_curves(y_true, y_scores):
    thresholds = np.linspace(0, 1, 50)
    precisions = []
    recalls = []

    for t in thresholds:
        preds = (y_scores >= t).astype(int)

        tp = np.sum((preds == 1) & (y_true == 1))
        fp = np.sum((preds == 1) & (y_true == 0))
        fn = np.sum((preds == 0) & (y_true == 1))

        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)

        precisions.append(precision)
        recalls.append(recall)

    plt.figure()
    plt.plot(thresholds, precisions, label="Precision")
    plt.plot(thresholds, recalls, label="Recall")
    plt.xlabel("Confidence Threshold")
    plt.ylabel("Score")
    plt.title("Confidence vs Precision/Recall")
    plt.legend()
    plt.grid()
    plt.show()


# -----------------------------
# 3. SEGMENTATION METRICS
# -----------------------------
def compute_iou_dice(pred, gt, num_classes):
    ious = []
    dices = []

    for cls in range(num_classes):
        pred_mask = (pred == cls)
        gt_mask = (gt == cls)

        intersection = np.logical_and(pred_mask, gt_mask).sum()
        union = np.logical_or(pred_mask, gt_mask).sum()

        iou = intersection / (union + 1e-6)
        dice = (2 * intersection) / (pred_mask.sum() + gt_mask.sum() + 1e-6)

        ious.append(iou)
        dices.append(dice)

    return ious, dices


# -----------------------------
# 4. BAR PLOTS (IoU / Dice)
# -----------------------------
def plot_bar(values, labels, title):
    plt.figure()
    plt.bar(labels, values)
    plt.title(title)
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    plt.show()


# -----------------------------
# 5. CONFUSION MATRIX (PIXEL LEVEL)
# -----------------------------
def plot_conf_matrix(pred, gt, num_classes):
    pred_flat = pred.flatten()
    gt_flat = gt.flatten()

    cm = confusion_matrix(gt_flat, pred_flat, labels=list(range(num_classes)))

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d")
    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.title("Pixel-wise Confusion Matrix")
    plt.show()


# -----------------------------
# 🔥 HOW TO USE THIS
# -----------------------------
"""
Assume you already have:

pred_mask → (H, W)
gt_mask → (H, W)
probs → (C, H, W)

"""

# Example usage:

def run_all_visualizations(pred_mask, gt_mask, probs, label_map):

    num_classes = len(label_map)

    # -------- SEGMENTATION --------
    ious, dices = compute_iou_dice(pred_mask, gt_mask, num_classes)

    labels = list(label_map.keys())

    plot_bar(ious, labels, "IoU per Class")
    plot_bar(dices, labels, "Dice per Class")

    plot_conf_matrix(pred_mask, gt_mask, num_classes)

    # -------- DETECTION STYLE (per class) --------
    for cls in range(1, num_classes):  # skip background
        gt_binary = (gt_mask == cls).astype(int)
        score_map = probs[cls]

        plot_pr_curve(gt_binary.flatten(), score_map.flatten(), f"class {cls}")
        plot_confidence_curves(gt_binary.flatten(), score_map.flatten())

In [3]:
import torch

# fake the missing function
def cmd_train(*args, **kwargs):
    pass

import __main__
__main__.cmd_train = cmd_train

# now load
ckpt = torch.load("runs/unet_transferLearn/best.pt", map_location="cpu")

clean_ckpt = {
    "model_state": ckpt["model_state"],
    "label_map": ckpt["label_map"],
    "image_size": ckpt["image_size"],
}

torch.save(clean_ckpt, "model_files/seg.pt")

RuntimeError: Parent directory model_files does not exist.